# Chronos-2 Interval Prediction Adaptation

## Load Models

In [1]:
from chronos import Chronos2Pipeline

model_vanilla_local = Chronos2Pipeline.from_pretrained(
    f"../Models/Chronos2/chronos2_model_local_vanilla",
    device_map="cpu"
)

model_vanilla_regional = Chronos2Pipeline.from_pretrained(
    f"../Models/Chronos2/chronos2_model_regional_vanilla",
    device_map="cpu"
)



Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

## Load Dataset
Adapted from Model Evaluation.ipynb

In [2]:
import pandas as pd

df_full = pd.read_parquet("../Data/EastAfrica Preprocessed/eastafrica.parquet")
df_full = df_full.drop(columns=df_full.filter(regex="Max").columns)

#drop the median columns if they still somehow exist:
df_full = df_full.drop(columns=df_full.filter(regex="Median").columns)
df_full = df_full.drop(columns=df_full.filter(regex="median").columns)

#type conversion: id to string. 
df_full["id"] = df_full["id"].astype("string")

#Remove rows with missing data
df_full = df_full.dropna(
    subset=["Armed_forces_personnel__of_total_labor_force_MSMILTOTLTFZS"]
)

#Remove bad Ids with suspect data:

# compute counts
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)

# get list of bad IDs
bad_ids = df_counts.query("count != 125")["id"].tolist()

# drop rows with those IDs
print("Before deletion:", df_full[df_full["id"].isin(bad_ids)].shape)
df_full.drop(df_full[df_full["id"].isin(bad_ids)].index, inplace=True)
print("After deletion:", df_full[df_full["id"].isin(bad_ids)].shape)


# recompute counts to verify
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)
df_full.reset_index(drop=True, inplace=True)


bad_ids_after = df_counts.query("count != 125")
print("Bad IDs in df_full:", bad_ids_after["id"].tolist())
print("Counts for bad IDs (df_full):")
print(bad_ids_after)

Before deletion: (2250, 53)
After deletion: (0, 53)
Bad IDs in df_full: []
Counts for bad IDs (df_full):
Empty DataFrame
Columns: [id, count]
Index: []


### Apply target variables

In [3]:
#Local:
df_full['target_conflict_indicator_local'] = (
    (df_full['LocalTotalFatalities'] >= 3) |
    (df_full['LocalMeanSeverity'] >= 3)
).astype(int)

#Regional:
df_full['target_conflict_indicator_regional'] = (
    (df_full['RegionalTotalFatalities'] >= 3) |
    (df_full['RegionalMeanSeverity'] >= 3)
).astype(int)

target_columns = ['target_conflict_indicator_local', 'target_conflict_indicator_regional']

### Split the full dataset by periodStart into Future and Context dataframes

In [13]:
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
all_period_starts
testing_periods = all_period_starts[:12]

testing_periods



[Timestamp('2025-07-31 00:00:00'),
 Timestamp('2025-07-03 00:00:00'),
 Timestamp('2025-06-05 00:00:00'),
 Timestamp('2025-05-08 00:00:00'),
 Timestamp('2025-04-10 00:00:00'),
 Timestamp('2025-03-13 00:00:00'),
 Timestamp('2025-02-13 00:00:00'),
 Timestamp('2025-01-16 00:00:00'),
 Timestamp('2024-12-19 00:00:00'),
 Timestamp('2024-11-21 00:00:00'),
 Timestamp('2024-10-24 00:00:00'),
 Timestamp('2024-09-26 00:00:00')]

In [35]:
cutoff_lower = '2024-09-26'
#cutoff_upper = df_test["periodStart"].max()

df_context_chronos = df_full[df_full["periodStart"] < cutoff_lower].copy()

df_future_chronos = df_full[
    (df_full["periodStart"] >= cutoff_lower)
].copy()

In [36]:
print("Context: ", min(df_context_chronos['periodStart']), ' - ', max(df_context_chronos['periodStart']))
print("Future: ", min(df_future_chronos['periodStart']), ' - ', max(df_future_chronos['periodStart']))

unique_dates = df_future_chronos["periodStart"].dt.date.unique()

print(unique_dates)

Context:  2016-01-28 00:00:00  -  2024-08-29 00:00:00
Future:  2024-09-26 00:00:00  -  2025-07-31 00:00:00
[datetime.date(2024, 9, 26) datetime.date(2024, 10, 24)
 datetime.date(2024, 11, 21) datetime.date(2024, 12, 19)
 datetime.date(2025, 1, 16) datetime.date(2025, 2, 13)
 datetime.date(2025, 3, 13) datetime.date(2025, 4, 10)
 datetime.date(2025, 5, 8) datetime.date(2025, 6, 5)
 datetime.date(2025, 7, 3) datetime.date(2025, 7, 31)]


### Reduce the size of the dataframes to 100 place names. 

In [41]:
# Get 100 random unique values from column x
sample_values = (
    df_full['id']
    .dropna()           # optional
    .unique()
)

sample_values = pd.Series(sample_values).sample(
    n=100,
    random_state=42     # for reproducibility
).tolist()

df_context_chronos = df_context_chronos[df_context_chronos['id'].isin(sample_values)].copy()
df_future_chronos = df_future_chronos[df_future_chronos['id'].isin(sample_values)].copy()


### Make basic predictions for 12 periods in the future. 

In [42]:
#drop_cols = [t for t in target_columns if t != target_column]



#Vanilla
pred_df_vanilla_local= model_vanilla_local.predict_df(
    df =df_context_chronos.drop(columns=['target_conflict_indicator_regional']).copy(),
    future_df=df_future_chronos.drop(columns = ['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(), #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target='target_conflict_indicator_local',  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
    )

In [43]:
pred_df_vanilla_local

,id,periodStart,target_name,predictions,0.1,0.5,0.9
0,Central_African_Republic_Bamatara,2024-09-26,target_conflict_indicator_local,0.000464,-0.000252,0.000464,0.001407
1,Central_African_Republic_Bamatara,2024-10-24,target_conflict_indicator_local,0.000529,-0.000401,0.000529,0.001419
2,Central_African_Republic_Bamatara,2024-11-21,target_conflict_indicator_local,0.000530,-0.000365,0.000530,0.001384
3,Central_African_Republic_Bamatara,2024-12-19,target_conflict_indicator_local,0.000547,-0.000369,0.000547,0.001430
4,Central_African_Republic_Bamatara,2025-01-16,target_conflict_indicator_local,0.000498,-0.000367,0.000498,0.001469
...,...,...,...,...,...,...,...
1231,Uganda_Tingey,2025-04-10,target_conflict_indicator_local,0.000680,-0.000249,0.000680,0.003253
1232,Uganda_Tingey,2025-05-08,target_conflict_indicator_local,0.000643,-0.000538,0.000643,0.003955
1233,Uganda_Tingey,2025-06-05,target_conflict_indicator_local,0.000562,-0.000747,0.000562,0.003483
1234,Uganda_Tingey,2025-07-03,target_conflict_indicator_local,0.000495,-0.000692,0.000495,0.002809


In [102]:
import pandas as pd
import numpy as np


def create_interval_targets_v2(
    df_predictions,
    df_actuals,
    actual_column_name,
    quartile,
    threshold
):
    df_predictions = (
        df_predictions
        .sort_values(["id", "periodStart"])
        .copy()
    )

    df_actuals = (
        df_actuals
        .sort_values(["id", "periodStart"])
        .copy()
    )

    df_predictions["positive"] = (
        df_predictions[quartile] >= threshold
    ).astype(int)

    records = []

    for id_value, pred_group in df_predictions.groupby("id"):

        pred_group = pred_group.sort_values("periodStart")

        # prediction interval targets
        p = pred_group["positive"].to_numpy()

        # start of forecast horizon
        forecast_start = pred_group["periodStart"].min()

        actual_group = (
            df_actuals.loc[df_actuals["id"] == id_value]
            .sort_values("periodStart")
            .reset_index(drop=True)
        )

        start_matches = actual_group.index[
            actual_group["periodStart"] == forecast_start
        ]

        if len(start_matches) == 0:
            raise ValueError(
                f"No matching actuals found for "
                f"id={id_value}, periodStart={forecast_start}"
            )

        start_idx = start_matches[0]

        actual_values = actual_group[actual_column_name].to_numpy()

        def actual_horizon(h):
            end_idx = start_idx + h

            if end_idx > len(actual_values):
                return pd.NA

            return actual_values[start_idx:end_idx].max()

        records.append({
            "id": id_value,
            "quartile": quartile,
            "threshold": threshold,

            # prediction targets
            "t+1": p[:1].max(),
            "t+3": p[:3].max(),
            "t+6": p[:6].max(),
            "t+12": p[:12].max(),

            # actual targets
            "actual_t+1": actual_horizon(1),
            "actual_t+3": actual_horizon(3),
            "actual_t+6": actual_horizon(6),
            "actual_t+12": actual_horizon(12),
        })

    return pd.DataFrame(records)

In [103]:
df_converted = create_interval_targets_v2(pred_df_vanilla_local, df_full, "target_conflict_indicator_local", "0.5", 0.5)
df_converted

,id,quartile,threshold,t+1,t+3,t+6,t+12,actual_t+1,actual_t+3,actual_t+6,actual_t+12
0,Central_African_Republic_Bamatara,0.5,0.5,0,0,0,0,0,0,0,0
1,Central_African_Republic_Bazanga,0.5,0.5,0,0,0,0,0,0,0,0
2,Central_African_Republic_Beloko,0.5,0.5,0,0,0,0,0,1,1,1
3,Central_African_Republic_Doko,0.5,0.5,0,0,0,0,0,0,0,1
4,Central_African_Republic_Dongo,0.5,0.5,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
98,Uganda_Muyanje,0.5,0.5,0,0,0,0,0,0,0,0
99,Uganda_Muyembe,0.5,0.5,0,0,0,0,0,0,0,0
100,Uganda_Nakapiripirit_Town_Council,0.5,0.5,0,0,0,0,0,0,1,1
101,Uganda_Padwot,0.5,0.5,0,0,0,0,0,0,0,0


### Test with a record with a positive value

In [104]:
quartile = "0.5"
threshold = 0.5

positive_ids = (
    pred_df_vanilla_local.loc[pred_df_vanilla_local[quartile] >= threshold, "id"]
      .unique()
)

test_id = positive_ids[0] 
test_id

'Ethiopia_Embully'

In [105]:
df =  pred_df_vanilla_local.loc[
        pred_df_vanilla_local["id"] == test_id].sort_values("periodStart")
df

,id,periodStart,target_name,predictions,0.1,0.5,0.9
276,Ethiopia_Embully,2024-09-26,target_conflict_indicator_local,0.893530,0.071563,0.893530,1.234803
277,Ethiopia_Embully,2024-10-24,target_conflict_indicator_local,0.701133,0.018579,0.701133,1.210508
278,Ethiopia_Embully,2024-11-21,target_conflict_indicator_local,0.811929,0.033105,0.811929,1.259310
279,Ethiopia_Embully,2024-12-19,target_conflict_indicator_local,0.174236,-0.011271,0.174236,0.960627
280,Ethiopia_Embully,2025-01-16,target_conflict_indicator_local,0.841973,0.061222,0.841973,1.231553
281,Ethiopia_Embully,2025-02-13,target_conflict_indicator_local,0.083093,-0.016099,0.083093,0.762995
282,Ethiopia_Embully,2025-03-13,target_conflict_indicator_local,0.836129,0.068326,0.836129,1.236947
283,Ethiopia_Embully,2025-04-10,target_conflict_indicator_local,0.221673,0.000409,0.221673,1.095984
284,Ethiopia_Embully,2025-05-08,target_conflict_indicator_local,0.217621,-0.005878,0.217621,1.086832
285,Ethiopia_Embully,2025-06-05,target_conflict_indicator_local,0.673961,0.035665,0.673961,1.183219


In [106]:
df =  df_converted.loc[
        df_converted["id"] == test_id]
df

,id,quartile,threshold,t+1,t+3,t+6,t+12,actual_t+1,actual_t+3,actual_t+6,actual_t+12
23,Ethiopia_Embully,0.5,0.5,1,1,1,1,1,1,1,1


In [95]:
df =  df_full.loc[
        df_full["id"] == test_id].sort_values("periodStart")
df[['id', 'periodStart', 'target_conflict_indicator_local']].tail(12)

,id,periodStart,target_conflict_indicator_local
1882689,Ethiopia_Embully,2024-09-26,1
1899325,Ethiopia_Embully,2024-10-24,1
1915961,Ethiopia_Embully,2024-11-21,1
1932597,Ethiopia_Embully,2024-12-19,0
1949233,Ethiopia_Embully,2025-01-16,1
1965869,Ethiopia_Embully,2025-02-13,0
1982505,Ethiopia_Embully,2025-03-13,1
1999141,Ethiopia_Embully,2025-04-10,0
2015777,Ethiopia_Embully,2025-05-08,0
2032413,Ethiopia_Embully,2025-06-05,0


## Compute Metrics from the Evaluated columns:

In [107]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


def evaluate_interval_predictions(df):
    horizons = ["1", "3", "6", "12"]

    results = {}

    for h in horizons:

        y_true = df[f"actual_t+{h}"]
        y_pred = df[f"t+{h}"]

        # Drop any missing values
        mask = y_true.notna() & y_pred.notna()

        y_true = y_true[mask].astype(int)
        y_pred = y_pred[mask].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        specificity = tn / (tn + fp) if (tn + fp) > 0 else None

        try:
            auc = roc_auc_score(y_true, y_pred)
        except ValueError:
            auc = None

        results[f"t+{h}"] = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Specificity": specificity,
            "Precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "AUC": auc,
            "Confusion Matrix": confusion_matrix(
                y_true,
                y_pred,
                labels=[0, 1]
            ),
            "Classification Report": classification_report(
                y_true,
                y_pred,
                zero_division=0
            )
        }

    return results

In [109]:
evaluate_interval_predictions(df_converted)

{'t+1': {'Accuracy': 0.9805825242718447,
  'Specificity': np.float64(1.0),
  'Precision': 1.0,
  'Recall': 0.6666666666666666,
  'F1': 0.8,
  'AUC': 0.8333333333333333,
  'Confusion Matrix': array([[97,  0],
         [ 2,  4]]),
  'Classification Report': '              precision    recall  f1-score   support\n\n           0       0.98      1.00      0.99        97\n           1       1.00      0.67      0.80         6\n\n    accuracy                           0.98       103\n   macro avg       0.99      0.83      0.89       103\nweighted avg       0.98      0.98      0.98       103\n'},
 't+3': {'Accuracy': 0.941747572815534,
  'Specificity': np.float64(1.0),
  'Precision': 1.0,
  'Recall': 0.5,
  'F1': 0.6666666666666666,
  'AUC': 0.75,
  'Confusion Matrix': array([[91,  0],
         [ 6,  6]]),
  'Classification Report': '              precision    recall  f1-score   support\n\n           0       0.94      1.00      0.97        91\n           1       1.00      0.50      0.67        